In [ ]:

# ============================================================
# IMPORTS
# ============================================================
# torch       : Core deep learning library. Provides Tensors (multi-dimensional
#               arrays that live on CPU/GPU), automatic differentiation
#               (autograd), and neural network primitives (nn.Module, Optimizers).
#
# urllib.request : Standard library module to fetch files from the internet
#                  without any third-party dependency.
#
# os          : Standard library module used here to check whether the
#               dataset file already exists on disk before re-downloading.
# ============================================================
import torch
import urllib.request
import os


In [ ]:

# ============================================================
# CELL 2 — Download the TinyShakespeare Dataset
# ============================================================
# TinyShakespeare is a ~1 MB plain-text file containing all of
# Shakespeare's plays concatenated together.  It is the canonical
# small dataset used to teach character-level language models because:
#   • It is small enough to train in minutes on a CPU.
#   • It has rich, recognisable structure (speaker names, dialogue,
#     acts/scenes) so human evaluation of quality is easy.
#
# We do an idempotency check with os.path.exists so that re-running
# the notebook never re-downloads the file unnecessarily.
# ============================================================

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

# Only download if the file is not already present in the working directory
if not os.path.exists("shakespeare.txt"):
    urllib.request.urlretrieve(url, "shakespeare.txt")
    print("Dataset downloaded successfully!")
else:
    print("Dataset already exists. Skipping download.")


Dataset downloaded successfully!


In [ ]:

# ============================================================
# CELL 3 — Build a Character-Level Tokenizer
# ============================================================
# WHAT IS TOKENIZATION?
#   Neural networks work with numbers, not strings.  Tokenization
#   is the process of converting raw text into a sequence of
#   integers (tokens) that the network can consume.
#
#   Modern LLMs (GPT-4, Claude) use sub-word tokenizers (BPE / SentencePiece)
#   where a token ≈ 3–4 characters on average.
#   We use the simplest possible scheme: every *individual character*
#   becomes one token.  This makes the vocabulary tiny (65 items) and
#   the code easy to follow, at the cost of longer sequences.
#
# VOCAB SIZE:
#   We read the full text, collect every unique character that appears,
#   sort them (for reproducibility), and call the count "vocab_size".
#   For TinyShakespeare: letters a-z, A-Z, digits, punctuation, newline,
#   space → 65 unique characters.
#
# MAPPING TABLES:
#   char_to_int  :  { 'a' → 0, 'b' → 1, … }   (used by the encoder)
#   int_to_char  :  { 0 → 'a', 1 → 'b', … }   (used by the decoder)
#
# ENCODE (str → list[int]):
#   "hello" → [35, 43, 50, 50, 53]
#   Each character is looked up in char_to_int independently.
#
# DECODE (list[int] → str):
#   [35, 43, 50, 50, 53] → "hello"
#   Each integer is looked up in int_to_char and joined into a string.
#
# PYTORCH TENSOR:
#   The whole text is converted once and stored as a 1-D torch.long tensor.
#   torch.long = int64 — required by nn.Embedding which expects integer indices.
# ============================================================

with open("shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"Dataset length: {len(text)} characters\n")

# --- Step 2: Extract Vocabulary ---
# sorted() ensures the same ordering every run (deterministic mapping).
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Unique characters (Vocab Size): {vocab_size}")
print(f"Vocab: {''.join(chars)}\n")

# --- Step 3: Build bi-directional lookup tables ---
# Enumerate assigns each character a unique integer index starting from 0.
char_to_int = {ch: i for i, ch in enumerate(chars)}   # encoder dict
int_to_char = {i: ch for i, ch in enumerate(chars)}   # decoder dict

# One-line lambda for convenience
# encode("hi") → [char_to_int['h'], char_to_int['i']] → [list of ints]
encode = lambda s: [char_to_int[c] for c in s]

# decode([35, 43]) → int_to_char[35] + int_to_char[43] → "he…"
decode = lambda l: "".join([int_to_char[i] for i in l])

# --- Quick sanity-check on the tokenizer ---
test_phrase = "hello mac"
encoded_samples = encode(test_phrase)
print(f"Testing Tokenizer on: '{test_phrase}'")
print(f"Encoded integers: {encoded_samples}")
print(f"Decoded back: '{decode(encoded_samples)}'\n")

# --- Step 4: Convert the entire text to a 1-D integer tensor ---
# dtype=torch.long (int64) is mandatory for use as indices in nn.Embedding.
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Data tensor shape: {data.shape}, Type: {data.dtype}")


Dataset length: 1115394 characters

Unique characters (Vocab Size): 65
Vocab: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz

Testing Tokenizer on: 'hello mac'
Encoded integers: [46, 43, 50, 50, 53, 1, 51, 39, 41]
Decoded back: 'hello mac'

Data tensor shape: torch.Size([1115394]), Type: torch.int64


In [ ]:

# ============================================================
# CELL 4 — Train/Validation Split & Batch Data Loader
# ============================================================
# TRAIN / VALIDATION SPLIT (90 / 10):
#   We never train on validation data.  If training loss keeps falling
#   but validation loss rises, the model is over-fitting (memorising
#   rather than generalising).  The 90/10 split is standard for this
#   dataset.
#
# KEY HYPERPARAMETERS:
#   block_size = 8  — The "context window".
#                     During training, the model is shown 8 characters
#                     and must predict character #9.
#                     Formally this is also called T (Time dimension).
#                     Why not make it bigger?  Larger T = more memory &
#                     compute. We start small for clarity.
#
#   batch_size = 4  — Number of independent sequences processed in one
#                     forward/backward pass.  Mini-batching parallelises
#                     computation on the GPU and also stabilises gradient
#                     updates (averaging over several sequences reduces
#                     the noise of stochastic gradient descent).
#
# HOW get_batch WORKS:
#   1. Pick `batch_size` random starting positions inside the dataset.
#   2. From each start i, slice:
#        x  = data[i   : i+block_size]   ← the *input* context (8 chars)
#        y  = data[i+1 : i+block_size+1] ← the *target* (same window shifted right by 1)
#   3. Stack slices into matrices:
#        xb  shape → (batch_size, block_size) = (4, 8)
#        yb  shape → (batch_size, block_size) = (4, 8)
#
#   WHY SHIFT BY 1?
#   For every position t in x, y[t] is "what comes next".  This gives
#   block_size training examples per sequence (the model learns to
#   predict the next char after 1 char, after 2 chars, …, after 8 chars).
#   Mathematically: given x[0..t-1], predict x[t] for t ∈ {1,…,T}.
# ============================================================

# Set seed for reproducibility
torch.manual_seed(1337)

# 1. Split into Train and Validation sets (90% / 10%)
n = int(0.9 * len(data))   # 90% of the total characters go to training
train_data = data[:n]
val_data = data[n:]

# 2. Define Hyperparameters for our Data Chunks
block_size = 8  # Context length: how many characters does the model read to predict the next?
batch_size = 4  # Batch size: how many sequences do we process in parallel?

# 3. Data Loader Function
def get_batch(split):
    """
    Returns a random mini-batch (x, y) of shape (batch_size, block_size).
    x : input token sequences
    y : target token sequences, shifted right by 1 position
    """
    data_set = train_data if split == 'train' else val_data

    # Generate `batch_size` random integer starting indices.
    # Upper bound is len(data_set) - block_size so the slice never goes out of bounds.
    ix = torch.randint(len(data_set) - block_size, (batch_size,))

    # Build the input matrix  x:  each row is data[i .. i+block_size-1]
    x = torch.stack([data_set[i:i+block_size] for i in ix])

    # Build the target matrix y:  each row is data[i+1 .. i+block_size],
    # i.e. the same window shifted one step to the right.
    y = torch.stack([data_set[i+1:i+block_size+1] for i in ix])
    return x, y

# Generate our very first training matrix batch
xb, yb = get_batch('train')

print("Inputs Tensor Matrix (xb) - shape:", xb.shape)
print(xb)
print("\nTargets Tensor Matrix (yb) - shape:", yb.shape)
print(yb)


Inputs Tensor Matrix (xb) - shape: torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])

Targets Tensor Matrix (yb) - shape: torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [ ]:

# ============================================================
# CELL 5 — Token + Positional Embedding Layer
# ============================================================
# PROBLEM: The raw integer tokens fed to the network carry no geometric
# meaning.  Token "35" and "36" are not "close" just because the integers
# are adjacent.  We must map each token to a rich, learnable vector.
#
# ── TOKEN EMBEDDING ───────────────────────────────────────────────────
# nn.Embedding(vocab_size, embedding_dim) creates a learnable matrix:
#
#     token_embedding_table  shape = [65, 32]   (V x C)
#
# Think of it as a dictionary: each of the 65 characters gets its own
# 32-dimensional floating-point vector (its "meaning" in vector space).
# During training these vectors are updated via backprop so that
# characters used in similar contexts end up near each other.
#
# Lookup:   idx (B, T)  →  embedding table  →  tok_emb (B, T, C)
#   For every integer in the (B,T) matrix, we fetch the corresponding
#   row from the 65×32 table, producing a 3-D cube.
#
# ── POSITIONAL EMBEDDING ─────────────────────────────────────────────
# Self-attention (next cell) is permutation-invariant by design — it
# treats the sequence as a *set*, not an *ordered list*.  We must inject
# the position of each token explicitly.
#
# nn.Embedding(block_size, embedding_dim) creates a second learnable matrix:
#
#     position_embedding_table  shape = [8, 32]   (T x C)
#
# We create position indices  [0, 1, 2, 3, 4, 5, 6, 7]  (one per time step)
# and look them up, producing pos_emb of shape (T, C) = (8, 32).
#
# ── COMBINING THEM ───────────────────────────────────────────────────
# Addition works because both tensors live in the same 32-D space and
# PyTorch broadcasting automatically repeats pos_emb across all B batches.
#
#   tok_emb (B, T, C)
# + pos_emb    (T, C)    ← broadcast to (B, T, C) automatically
# ──────────────────
# =   x     (B, T, C)    ← each token now encodes both WHAT it is
#                            and WHERE it sits in the sequence
#
# With our demo values:  B=4, T=8, C=32  →  output shape = (4, 8, 32)
# ============================================================

import torch
import torch.nn as nn

# 1. Define our structural dimensions
vocab_size = 65      # Number of unique characters in our alphabet
embedding_dim = 32   # Size of our geometric vector space (C)
block_size = 8       # Max context length (T)

class MiniEmbeddingLayer(nn.Module):
    def __init__(self):
        super().__init__()
        # Token Embedding Table: A 65 x 32 learnable weight matrix.
        # Each row is the learned "identity vector" of one character.
        self.token_embedding_table = nn.Embedding(vocab_size, embedding_dim)

        # Position Embedding Table: An 8 x 32 learnable weight matrix.
        # Each row is the learned "position vector" for slot 0..7.
        self.position_embedding_table = nn.Embedding(block_size, embedding_dim)

    def forward(self, idx):
        # idx is our batch matrix of integer tokens, shape: (B, T) [Batch, Time]
        B, T = idx.shape

        # Token lookup: each integer in (B,T) maps to its C-dim vector.
        # Result: tok_emb shape = (B, T, C)
        tok_emb = self.token_embedding_table(idx)

        # Build position indices: [0, 1, 2, ..., T-1]
        # These are the "slot numbers" for each position in the sequence.
        pos_indices = torch.arange(T, device=idx.device)   # shape: (T,)

        # Position lookup: each index maps to its C-dim position vector.
        # Result: pos_emb shape = (T, C)
        pos_emb = self.position_embedding_table(pos_indices)

        # Element-wise addition: broadcast (T, C) → (B, T, C) and add.
        # After this, x[b, t, :] = "who I am" + "where I am" in 32-D space.
        x = tok_emb + pos_emb   # shape: (B, T, C) = (4, 8, 32)
        return x

# --- Instantiate and test ---
embedding_layer = MiniEmbeddingLayer()

# Pass our real training batch (xb) from Cell 4 through this layer
embedded_output = embedding_layer(xb)

print(f"Original Input Shape (Batch, Block): {xb.shape}")
print(f"Embedded Output Shape (Batch, Block, Embed_Dim): {embedded_output.shape}")
print("\nLet's peek at the 32-dimensional vector for the very first character in our batch:")
print(embedded_output[0, 0, :])


Original Input Shape (Batch, Block): torch.Size([4, 8])
Embedded Output Shape (Batch, Block, Embed_Dim): torch.Size([4, 8, 32])

Let's peek at the 32-dimensional vector for the very first character in our batch:
tensor([-1.8032,  0.3709, -1.8922,  0.1120, -0.9482,  3.8708,  0.6438, -0.6854,
        -2.0068,  0.2788, -0.3158, -0.5066,  1.5468, -2.2322,  0.1589,  1.8112,
         0.4024,  1.0583, -0.1508, -0.9306,  1.0170, -0.1050,  0.0821,  0.0320,
        -0.8284,  2.6779,  0.0097, -1.0060, -0.8755, -1.4434, -0.1723,  1.9194],
       grad_fn=<SliceBackward0>)


In [ ]:

# ============================================================
# CELL 6 — Scaled Dot-Product Self-Attention (Single Head)
# ============================================================
#
# THE CORE QUESTION ATTENTION ANSWERS:
#   "For token at position t, how much should I attend to every
#    other token at position s ≤ t when building my representation?"
#
# ── THE THREE LEARNABLE PROJECTIONS ──────────────────────────────────
#
#   Query  Q = x · W_Q    shape: (B, T, head_size)
#   Key    K = x · W_K    shape: (B, T, head_size)
#   Value  V = x · W_V    shape: (B, T, head_size)
#
#   Each is a linear transformation of the input x (B, T, C):
#     W_Q, W_K, W_V  are weight matrices of shape (C, head_size).
#   These weights are LEARNABLE — the model discovers during training
#   what "questions", "descriptions", and "content" mean.
#
#   Intuition:
#     Q  = "what am I looking for?"
#     K  = "what do I contain / advertise?"
#     V  = "what value will I pass if selected?"
#
# ── ATTENTION SCORES (AFFINITIES) ────────────────────────────────────
#
#   wei = Q · Kᵀ / sqrt(head_size)     shape: (B, T, T)
#
#   Q · Kᵀ computes the dot product between every pair of (query, key)
#   vectors.  A high dot product means the query and key are aligned
#   → the model should pay a lot of attention to that position.
#
#   Scaling by 1/sqrt(head_size) is crucial:
#     • Without it, the dot products grow large in magnitude as head_size
#       increases.  Large values → near-zero gradients after softmax
#       (softmax saturates into a one-hot distribution).
#     • Dividing by sqrt(head_size) keeps the variance of wei ≈ 1,
#       making gradients healthy throughout training.
#
# ── CAUSAL MASK (DECODER-ONLY, NO FUTURE PEEKING) ───────────────────
#
#   self.tril = lower-triangular matrix of 1s:
#
#       [[1, 0, 0, 0, 0, 0, 0, 0],
#        [1, 1, 0, 0, 0, 0, 0, 0],
#        [1, 1, 1, 0, 0, 0, 0, 0],
#        ...                     ]
#
#   Positions where tril == 0 correspond to future tokens.
#   We replace those scores with -∞ BEFORE applying softmax.
#   After softmax, exp(-∞) = 0, so those positions get 0 probability.
#   Token at position t can only attend to tokens at positions 0..t.
#
#   This is what makes this an *autoregressive* (causal) model:
#   it can only look at the past, never the future.
#
# ── SOFTMAX (NORMALISE TO PROBABILITIES) ─────────────────────────────
#
#   wei = softmax(wei, dim=-1)    (still shape (B, T, T))
#
#   Each row of the (T×T) matrix sums to 1.0.
#   Row t now represents a probability distribution over which past
#   tokens token t "chooses to listen to".
#
# ── WEIGHTED AGGREGATION ─────────────────────────────────────────────
#
#   out = wei · V     shape: (B, T, T) × (B, T, head_size) → (B, T, head_size)
#
#   Each token's output is a weighted sum of all Value vectors,
#   where the weights are the attention probabilities.
#   If token t attends mostly to token s, its output will be
#   dominated by V[s].
#
# SUMMARY OF SHAPES (with B=4, T=8, C=32, head_size=16):
#   x      : (4,  8, 32)
#   Q,K,V  : (4,  8, 16)
#   wei    : (4,  8,  8)   ← attention weight matrix
#   out    : (4,  8, 16)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

# Hyperparameters from previous steps
embedding_dim = 32   # C — dimension of each token embedding
head_size = 16       # Dimensionality of Q/K/V projections

class SingleHeadAttention(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        # Three independent linear projections (no bias for cleaner math)
        self.query = nn.Linear(embedding_dim, head_size, bias=False)
        self.key   = nn.Linear(embedding_dim, head_size, bias=False)
        self.value = nn.Linear(embedding_dim, head_size, bias=False)

        # Lower-triangular mask — registered as a buffer so it is saved
        # with the model state but not treated as a trainable parameter.
        # Shape: (block_size, block_size) = (8, 8)
        self.register_buffer('tril', torch.tril(torch.ones(8, 8)))

    def forward(self, x):
        # x is the embedded input tensor of shape (B, T, C) -> [4, 8, 32]
        B, T, C = x.shape

        # ── Step 1: Compute Query, Key, Value ──────────────────────────
        # Each projects C-dimensional embeddings down to head_size dimensions.
        q = self.query(x)  # (B, T, C) × (C, head_size)  →  (B, T, head_size)  [4, 8, 16]
        k = self.key(x)    # (B, T, C) × (C, head_size)  →  (B, T, head_size)  [4, 8, 16]
        v = self.value(x)  # (B, T, C) × (C, head_size)  →  (B, T, head_size)  [4, 8, 16]

        # ── Step 2: Scaled Dot-Product Attention Scores ────────────────
        # Transpose K's last two dims: (B, T, head_size) → (B, head_size, T)
        # Batched matmul: (B, T, head_size) × (B, head_size, T) → (B, T, T)
        # Divide by sqrt(head_size) to normalise variance.
        wei = q @ k.transpose(-2, -1) * (head_size ** -0.5)  # shape: [4, 8, 8]

        # ── Step 3: Apply Causal Mask ──────────────────────────────────
        # Set upper-triangular (future) positions to -infinity.
        # [:T, :T] slices in case T < block_size (e.g. at sequence start).
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # ── Step 4: Softmax (row-wise normalisation to probabilities) ──
        # dim=-1 applies softmax along the last axis (the "key" axis).
        # Each row sums to 1; -inf entries become 0 after exp.
        wei = F.softmax(wei, dim=-1)   # shape: [4, 8, 8]

        # ── Step 5: Weighted Sum of Values ────────────────────────────
        # (B, T, T) × (B, T, head_size) → (B, T, head_size)
        # Each token's output = weighted combination of all V vectors.
        out = wei @ v   # shape: [4, 8, 16]

        return out, wei

# --- Instantiate and inspect ---
attention_head = SingleHeadAttention(head_size=head_size)

# Pass the embedded output from Cell 5 into this attention head
attention_out, attention_weights = attention_head(embedded_output)

print(f"Input Tensor Shape: {embedded_output.shape}")
print(f"Attention Output Shape: {attention_out.shape}")
print(f"Attention Weights Matrix Shape (B, T, T): {attention_weights.shape}")

print("\nLet's visually inspect the 8x8 Attention Weights Matrix for the first batch row:")
# Each row is a probability distribution over the 8 past positions.
# Lower-left triangle has non-zero values; upper-right is all zeros (causal mask).
print(torch.round(attention_weights[0], decimals=2))


Input Tensor Shape: torch.Size([4, 8, 32])
Attention Output Shape: torch.Size([4, 8, 16])
Attention Weights Matrix Shape (B, T, T): torch.Size([4, 8, 8])

Let's visually inspect the 8x8 Attention Weights Matrix for the first batch row:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2400, 0.7600, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3100, 0.5700, 0.1300, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1900, 0.3100, 0.4300, 0.0800, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1600, 0.0900, 0.2400, 0.1300, 0.3800, 0.0000, 0.0000, 0.0000],
        [0.1900, 0.1700, 0.2000, 0.1600, 0.0700, 0.2000, 0.0000, 0.0000],
        [0.0700, 0.1000, 0.0400, 0.3300, 0.0900, 0.1800, 0.1900, 0.0000],
        [0.0300, 0.2500, 0.1500, 0.1300, 0.0400, 0.0300, 0.1200, 0.2400]],
       grad_fn=<RoundBackward1>)


# Complete code

In [ ]:

# ============================================================
# CELL 8 — Full Mini-GPT: Architecture + Training Loop
# ============================================================
#
# This cell assembles and trains a complete decoder-only Transformer
# (the same fundamental architecture as GPT-2/3, just much smaller).
#
# ── ARCHITECTURE OVERVIEW ─────────────────────────────────────────────
#
#  Input tokens (B, T)
#      ↓  token embedding  +  positional embedding    → (B, T, n_embd)
#      ↓  N × Transformer Block
#          ├─ LayerNorm  →  MultiHeadAttention  →  residual add
#          └─ LayerNorm  →  FeedForward         →  residual add
#      ↓  final LayerNorm
#      ↓  linear head                            → (B, T, vocab_size)
#      ↓  cross-entropy loss
#
# ── HYPERPARAMETERS ──────────────────────────────────────────────────
#   batch_size  = 64   → 64 sequences in every mini-batch (was 4)
#   block_size  = 64   → 64-character context window (was 8)
#   n_embd      = 128  → each token is a 128-D vector
#   n_head      = 4    → 4 attention heads; each head_size = 128/4 = 32
#   n_layer     = 4    → 4 stacked Transformer blocks
#   lr          = 1e-3 → AdamW learning rate
#   max_iters   = 1500 → total gradient steps
#
# ── HEAD (Single Attention Head) ─────────────────────────────────────
#   Same as Cell 6 but uses n_embd and a variable head_size.
#   Returns the weighted-sum output (B, T, head_size).
#
# ── MULTI-HEAD ATTENTION ─────────────────────────────────────────────
#   Runs `num_heads` independent attention heads IN PARALLEL and
#   concatenates their outputs along the channel dimension:
#
#     [head_1 | head_2 | head_3 | head_4]   dim=-1 concat
#
#   Concatenated output shape: (B, T, num_heads * head_size) = (B, T, n_embd)
#   A final linear projection (proj) mixes the information across heads.
#
#   Why multiple heads?  Each head can specialise in a different type of
#   relationship (syntax, rhyme, subject-verb, etc.).  Concatenation
#   then lets the next layer access all of those representations.
#
# ── FEED-FORWARD NETWORK (FFN) ───────────────────────────────────────
#   Applied independently to each token position after attention:
#
#     FFN(x) = ReLU( x · W1 + b1 ) · W2 + b2
#
#   W1: n_embd → 4*n_embd  (expand)
#   W2: 4*n_embd → n_embd  (contract)
#
#   The 4× expansion comes from the original "Attention is All You Need"
#   paper.  It gives the model capacity to do non-linear computation
#   per token after the attention aggregation step.
#
# ── TRANSFORMER BLOCK ────────────────────────────────────────────────
#   Pre-norm architecture (LayerNorm BEFORE each sub-layer):
#
#     x = x + MHA(LayerNorm(x))   ← residual connection
#     x = x + FFN(LayerNorm(x))   ← residual connection
#
#   Residual connections ("skip connections") let gradients flow
#   directly from the loss to early layers without vanishing.
#   Pre-norm stabilises training vs. the original post-norm design.
#
# ── FULL MODEL FORWARD PASS ──────────────────────────────────────────
#   1. Embed tokens + positions → (B, T, n_embd)
#   2. Pass through N Blocks   → (B, T, n_embd)
#   3. Final LayerNorm         → (B, T, n_embd)
#   4. Linear "language model head" → logits (B, T, vocab_size)
#   5. If targets provided, flatten to (B*T, vocab_size) and compute
#      cross-entropy loss:
#
#        L = -sum_t log P(target_t | context)
#
# ── AUTOREGRESSIVE GENERATION ────────────────────────────────────────
#   The generate() method predicts one token at a time:
#     1. Crop the sequence to the last block_size tokens.
#     2. Forward pass → logits for the last time step.
#     3. Softmax to get next-token probabilities.
#     4. Sample one token from that distribution (multinomial sampling).
#     5. Append the token and repeat.
#
# ── TRAINING LOOP ────────────────────────────────────────────────────
#   Standard mini-batch stochastic gradient descent with AdamW:
#     1. Sample a batch.
#     2. Forward pass → compute loss.
#     3. Zero old gradients.
#     4. Backward pass → compute new gradients.
#     5. Optimizer step → update weights.
#   Every `eval_interval` steps we estimate train and val loss (without
#   accumulating gradients, hence @torch.no_grad).
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

# --- 1. Define Global Hyperparameters ---
batch_size = 64        # Increased from 4 to maximise processing speed
block_size = 64        # Increased from 8 so the model sees full sentences
max_iters = 1500       # How many total training steps to run
eval_interval = 200    # How often to check validation loss
learning_rate = 1e-3   # Step size for AdamW optimiser

# Automatically pick the best available compute device:
#   'cuda' → NVIDIA GPU | 'mps' → Apple Silicon GPU | 'cpu' → fallback
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')

n_embd = 128    # Embedding dimension (C)
n_head = 4      # Number of attention heads; each head_size = n_embd // n_head = 32
n_layer = 4     # Number of stacked Transformer blocks

print(f"Using device: {device.upper()}")

# --- 2. Re-verify Data Pipeline from Cells 3 & 4 ---
# Re-create train/val split to ensure variables are accessible in this cell.
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    """Sample a random (B, T) mini-batch and move it to the target device."""
    data_set = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_set) - block_size, (batch_size,))
    x = torch.stack([data_set[i:i+block_size] for i in ix])
    y = torch.stack([data_set[i+1:i+block_size+1] for i in ix])
    # Move tensors to GPU/MPS if available for accelerated computation
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()   # Disable gradient tracking to save memory during evaluation
def estimate_loss(model):
    """
    Estimate average loss on both train and val splits using 50 mini-batches each.
    Uses model.eval() to disable dropout / batch-norm running stats during eval.
    """
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(50)
        for k in range(50):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()   # Switch back to training mode
    return out

# --- 3. Build Transformer Building Blocks ---

class Head(nn.Module):
    """One single head of causal self-attention."""
    def __init__(self, head_size):
        super().__init__()
        # Three learnable (no-bias) projections from n_embd → head_size
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # Non-trainable lower-triangular causal mask, shape: (block_size, block_size)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)

        # Scaled dot-product attention scores: (B, T, T)
        # Dividing by sqrt(C) keeps variance stable (prevents softmax saturation)
        wei = q @ k.transpose(-2, -1) * (C**-0.5)

        # Apply causal mask: future positions → -inf → 0 after softmax
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # Row-wise softmax → attention probability distribution (B, T, T)
        wei = F.softmax(wei, dim=-1)

        v = self.value(x)  # (B, T, head_size)
        # Aggregate values weighted by attention probabilities
        out = wei @ v      # (B, T, head_size)
        return out

class MultiHeadAttention(nn.Module):
    """
    Multiple heads of self-attention running in parallel.
    Each head independently attends to different aspects of the sequence.
    Outputs are concatenated and linearly projected back to n_embd.
    """
    def __init__(self, num_heads, head_size):
        super().__init__()
        # Create num_heads independent Head modules in a list
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # Projection: maps concatenated output (n_embd) back to n_embd
        # This mixing layer allows heads to communicate after concatenation.
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        # Run all heads in parallel and concatenate along the channel dimension
        # Each head produces (B, T, head_size); cat → (B, T, num_heads * head_size) = (B, T, n_embd)
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        # Final linear mix across the concatenated head outputs
        out = self.proj(out)   # still (B, T, n_embd)
        return out

class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network applied independently to each token.
    Architecture: Linear(n_embd → 4*n_embd) → ReLU → Linear(4*n_embd → n_embd)
    The 4× expansion ratio follows the original Transformer paper.
    """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),   # expand: more capacity for non-linear computation
            nn.ReLU(),                         # element-wise non-linearity
            nn.Linear(4 * n_embd, n_embd),    # contract back to model dimension
        )

    def forward(self, x):
        return self.net(x)   # applied identically to every token position

class Block(nn.Module):
    """
    One Transformer Block = Self-Attention + Feed-Forward with residual connections.

    Uses Pre-LayerNorm (norm BEFORE sub-layer) for training stability:
        x = x + MHA(LN(x))
        x = x + FFN(LN(x))

    Residual connections let gradients bypass the sub-layers and flow
    directly back to earlier layers, preventing vanishing gradients.
    """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head   # 128 // 4 = 32 per head
        self.sa   = MultiHeadAttention(n_head, head_size)   # self-attention
        self.ffwd = FeedForward(n_embd)                     # position-wise FFN
        self.ln1  = nn.LayerNorm(n_embd)                    # norm before attention
        self.ln2  = nn.LayerNorm(n_embd)                    # norm before FFN

    def forward(self, x):
        # Residual: x + Attention(LayerNorm(x))
        x = x + self.sa(self.ln1(x))
        # Residual: x + FFN(LayerNorm(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# --- 4. Assemble the Full Language Model ---

class MiniLanguageModel(nn.Module):
    """
    Decoder-only Transformer Language Model.
    Produces a probability distribution over the next token for every
    position in the input sequence.
    """
    def __init__(self):
        super().__init__()
        # Learnable token embedding lookup table: shape (vocab_size, n_embd)
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # Learnable position embedding lookup table: shape (block_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # Stack of n_layer Transformer blocks (communication + computation layers)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        # Final LayerNorm stabilises activations before the output projection
        self.ln_f = nn.LayerNorm(n_embd)
        # Language model head: projects n_embd back to vocabulary size (65 logits)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # ── Embedding stage ──
        tok_emb = self.token_embedding_table(idx)                        # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))  # (T, n_embd)
        x = tok_emb + pos_emb                                            # (B, T, n_embd)

        # ── Transformer blocks ──
        x = self.blocks(x)   # 4 sequential blocks; shape stays (B, T, n_embd)

        # ── Final LayerNorm ──
        x = self.ln_f(x)     # (B, T, n_embd)

        # ── Language model head → logits ──
        logits = self.lm_head(x)   # (B, T, vocab_size)

        # ── Loss computation (optional) ──
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            # Flatten to 2-D for cross_entropy: (B*T, C) vs (B*T,)
            logits  = logits.view(B*T, C)
            targets = targets.view(B*T)
            # Cross-entropy loss = -log(softmax(logits)[target])
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        """
        Autoregressively generate `max_new_tokens` new tokens one at a time.
        idx: (B, T) starting token sequence (context/prompt).
        Returns: (B, T + max_new_tokens) token sequence.
        """
        for _ in range(max_new_tokens):
            # Crop context to the last block_size tokens (avoid exceeding position table)
            idx_cond = idx[:, -block_size:]

            # Forward pass — we only need the logits, not the loss
            logits, loss = self(idx_cond)

            # Focus on the LAST time step (the next-token prediction)
            logits = logits[:, -1, :]   # (B, vocab_size)

            # Convert raw logits to a probability distribution
            probs = F.softmax(logits, dim=-1)   # (B, vocab_size)

            # Sample one token index from the distribution (stochastic generation)
            idx_next = torch.multinomial(probs, num_samples=1)   # (B, 1)

            # Append the sampled token to the running context
            idx = torch.cat((idx, idx_next), dim=1)   # (B, T+1)

        return idx

# --- 5. Instantiate, Count Parameters, Train ---

model = MiniLanguageModel()
m = model.to(device)   # Move all parameters to the chosen device

# Count total trainable parameters (~1 million for these settings)
total_params = sum(p.numel() for p in m.parameters())
print(f"Total Model Parameters: {total_params:,}")

# AdamW: Adam with weight decay (L2 regularisation on weights, not biases).
# Weight decay prevents overfitting by penalising large weight magnitudes.
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("Starting training loop... Watch the loss drop!")
for iter in range(max_iters):

    # Periodically evaluate on both train and val to monitor for overfitting
    if iter % eval_interval == 0:
        losses = estimate_loss(m)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # --- One gradient descent step ---
    xb, yb = get_batch('train')         # Sample a fresh random mini-batch

    logits, loss = model(xb, yb)        # Forward pass → compute loss

    optimizer.zero_grad(set_to_none=True)  # Clear stale gradients (set_to_none saves memory)
    loss.backward()                         # Backward pass → compute ∂loss/∂weights
    optimizer.step()                        # Update weights: w ← w - lr * grad

print(f"Training Complete! Final Loss: {loss.item():.4f}")


Using device: MPS
Total Model Parameters: 816,705
Starting training loop... Watch the loss drop!
step 0: train loss 4.3555, val loss 4.3600
step 200: train loss 2.2524, val loss 2.2758
step 400: train loss 1.9633, val loss 2.0436
step 600: train loss 1.7809, val loss 1.9150
step 800: train loss 1.6766, val loss 1.8294
step 1000: train loss 1.5988, val loss 1.7680
step 1200: train loss 1.5483, val loss 1.7131
step 1400: train loss 1.5118, val loss 1.6898
Training Complete! Final Loss: 1.4813


In [ ]:

# ============================================================
# CELL 9 — Unconditional Text Generation (Blank Prompt)
# ============================================================
# We test the trained model by letting it generate freely from a
# single "seed" token (index 0, which maps to '\n').
#
# A (1, 1) context tensor is the minimal valid input:
#   • Batch dimension = 1  (one generation stream)
#   • Time dimension  = 1  (one initial token)
#
# generate() then autoregressively extends this by 500 tokens:
#   At each step the model predicts P(next_token | current_context)
#   and samples one token from that distribution.
#
# If the model has trained well, the output should resemble Shakespearean
# prose — proper speaker names followed by dialogue, with plausible
# English words, even if the content is nonsensical.
# ============================================================

# 1. Start with a clean slate: feed a 1x1 matrix containing index 0 (usually a newline/space)
# We send it to the same device (CPU/GPU) where our model lives
context = torch.zeros((1, 1), dtype=torch.long, device=device)

# 2. Ask the model to generate 500 characters step-by-step (autoregressively)
generated_tokens = m.generate(context, max_new_tokens=500)

# 3. Take the 2D tensor of generated integer IDs, extract the flat list, and decode to text
# generated_tokens[0] selects the first (and only) batch row → 1-D list of ints
print("--- GENERATED SHAKESPEARE TEXT ---")
print(decode(generated_tokens[0].tolist()))
print("----------------------------------")


--- GENERATED SHAKESPEARE TEXT ---


Nay, my lord, charist expecured
in him our king I centage a
prourinted my thunders? fearion thy scorns;
How man you will home by servicials
Toke a lithtchinion, away! See maying could strel with us;
I may a prishase a quarderal of your tralls;
And proging he for he givn you that you deatle
Thau tis lastruman
And your bloody.

First Mayor:
But herminous down.

CORIOLANUS:
And bracher.

DAUTIUS:
Thou Topped retords it his and fair from
You man, this King Becomersaby to call,
Kate-charged the kind
----------------------------------


In [ ]:

# ============================================================
# CELL 10 — Save Model Weights to Disk
# ============================================================
# torch.save(model.state_dict(), path) serialises ONLY the learnable
# parameters (weights + biases) as a Python dict, not the entire
# model class definition.  This is the recommended PyTorch practice:
#   • Lightweight: saves just numbers, not code.
#   • Portable: can be loaded into any model that shares the same
#     architecture, even across Python versions.
#   • To reload:
#       m2 = MiniLanguageModel()
#       m2.load_state_dict(torch.load('mini_shakespeare_model.pt'))
#       m2.eval()
# ============================================================

# Save the trained weights to a file named 'mini_shakespeare_model.pt'
torch.save(model.state_dict(), 'mini_shakespeare_model.pt')
print("Model frozen and saved successfully!")


Model frozen and saved successfully!


In [ ]:

# ============================================================
# CELL 11 — Conditional (Prompt-Based) Text Generation
# ============================================================
# Instead of starting from a blank token, we "prime" the model
# with a real text prefix.  This is called conditional generation:
#   P(continuation | prompt)
#
# The prompt "ROMEO:" gives the model strong context:
#   • It sees the speaker's name, so it "knows" it's generating
#     dialogue for that character.
#   • This tends to produce more coherent, on-topic continuations
#     than the blank-prompt baseline from Cell 9.
#
# Steps:
#   1. Encode the prompt string to a list of integers.
#   2. Reshape into a (1, len(prompt)) 2-D tensor and move to device.
#   3. Pass into generate() — the model extends the sequence by 300 tokens.
#   4. Decode the full output (prompt + generated tokens) back to text.
# ============================================================

# 1. Take a custom text prompt and convert it to a list of integers
prompt = "ROMEO:"
encoded_prompt = encode(prompt)   # e.g. [44, 27, 25, 17, 27, 10]

# 2. Wrap in a 2-D Tensor: shape (1, len(prompt)) = (1, 6)
# The model always expects a batch dimension even if batch_size=1.
context_tensor = torch.tensor([encoded_prompt], dtype=torch.long, device=device)

# 3. Generate 300 new tokens autoregressively after the prompt
generated_output = m.generate(context_tensor, max_new_tokens=300)

# 4. Decode the entire output — includes the original prompt tokens followed by
# all newly generated tokens.
print("--- PROMPT COMPLETED BY MODEL ---")
print(decode(generated_output[0].tolist()))
print("---------------------------------")


--- PROMPT COMPLETED BY MODEL ---
ROMEO:
Madam: the king unpility this horself to her.

MUCIOTIO:
Ay, I hum hondry toucher stainst in your countemp,
Indulister to taums soldeps wore o' the faults from breathen?
Shall any yeart should a tow and brother.
To sit Lord Haven. Presures not, and the sui,
Afferishmas were keep thumbertincaed up i
---------------------------------


In [ ]:

# ============================================================
# CELL 12 — Introspect the Learned Embedding Space
# ============================================================
# WHAT ARE WE DOING?
#   After training, the token embedding table is a 65×128 matrix
#   of floating-point numbers.  Each row is the "meaning vector"
#   that the model has learned to associate with one character.
#
#   Characters that appear in similar contexts during training will
#   have been pushed toward similar regions of this 128-D space
#   by gradient descent.  For example:
#     • 'a', 'e', 'o' (vowels) may cluster together.
#     • 'A', 'B', 'C' (uppercase) may cluster together.
#     • '\n' and ' ' (whitespace) may cluster together.
#
# COSINE SIMILARITY:
#   A standard way to measure the angle between two vectors,
#   regardless of their magnitude:
#
#       cos(θ) = (u · v) / (‖u‖ · ‖v‖)
#
#   Returns a value in [-1, 1]:
#     +1  → perfectly aligned (same direction)
#      0  → orthogonal (no relationship)
#     -1  → perfectly opposed
#
#   We compute this between the target character's vector and every
#   other character's vector in one vectorised operation.
#
# NOTE: This is a *post-hoc* analysis — we are inspecting what the
# model learned, not training it.  Always call model.eval() (or
# wrap in torch.no_grad()) when performing inference to suppress
# unnecessary gradient tracking.
# ============================================================

# 1. Extract the raw weight matrix from the token embedding layer
# Shape: [vocab_size, n_embd] = [65, 128]
embedding_matrix = m.token_embedding_table.weight.data

# 2. Pick a target character to analyse
target_char = 'e'
target_idx = char_to_int[target_char]
target_vector = embedding_matrix[target_idx]   # shape: (128,) — the 128-D vector for 'e'

# 3. Compute cosine similarity between the target vector and EVERY other row.
# F.cosine_similarity broadcasts: (1, 128) vs (65, 128) → (65,) similarity scores.
# unsqueeze(0) adds a batch dimension so the function handles the comparison.
similarities = F.cosine_similarity(target_vector.unsqueeze(0), embedding_matrix, dim=1)

# 4. Pair each character with its similarity score and sort descending
char_scores = [(int_to_char[i], similarities[i].item()) for i in range(vocab_size)]
char_scores = sorted(char_scores, key=lambda x: x[1], reverse=True)

# 5. Print the top 5 nearest geometric neighbours (skip rank 0, which is 'e' itself = 1.0)
print(f"Characters most geometrically similar to '{target_char}' in vector space:")
for char, score in char_scores[1:6]:
    # Replace non-printable characters with readable labels
    display_char = '\\n' if char == '\n' else ('[SPACE]' if char == ' ' else char)
    print(f" Character: {display_char}  | Similarity Score: {score:.4f}")


Characters most geometrically similar to 'e' in vector space:
 Character: a  | Similarity Score: 0.1956
 Character: .  | Similarity Score: 0.1178
 Character: D  | Similarity Score: 0.1125
 Character: H  | Similarity Score: 0.1096
 Character: x  | Similarity Score: 0.1062


In [ ]:

# ============================================================
# CELL 13 — Robustness Test: Recovery from Gibberish Input
# ============================================================
# This is a stress-test for the model's learned language prior.
#
# We feed it a nonsense prompt of 6 'x' characters ("xxxxxx").
# 'x' is a rare character in Shakespeare — it appears almost
# exclusively in words like "execute", "exile", or as a letter
# in names — so a long run of 'x's is completely out-of-distribution.
#
# Despite this, the model should "recover" after a few steps
# because:
#   1. The 'x' tokens are encoded into 128-D vectors and pass through
#      the attention blocks just like any other token.
#   2. The model has no explicit memory of the prompt — it just
#      predicts the next character based on the current context.
#   3. As the generated characters accumulate (spaces, vowels, etc.)
#      they begin to dominate the context window, steering the
#      distribution back toward valid English patterns.
#
# Generating 1000 tokens gives a clear before-and-after: the early
# output may be erratic but it should quickly settle into something
# resembling Shakespearean text.
# ============================================================

garbage_prompt = "xxxxxx"   # 6 rare/unusual characters — an out-of-distribution prompt

# Encode and wrap as (1, 6) tensor, sent to the model's device
context_garbage = torch.tensor([encode(garbage_prompt)], dtype=torch.long, device=device)

# Generate 1000 tokens — long enough to observe recovery behaviour
garbage_output = m.generate(context_garbage, max_new_tokens=1000)

print("--- RECOVERY FROM GIBBERISH ---")
print(decode(garbage_output[0].tolist()))
print("-------------------------------")


--- RECOVERY FROM GIBBERISH ---
xxxxxxated play of controun
To seen,
Even his sour to distredawest of Foresuice:
There to so me, guiles thou his, that I lave:
And my hereason
I will quainsay a look. And thinked will you thus.
By will nor curpt leaken you Vicencent of come,
And thou I muded remain a cive king himfully no
cominionting rin of and and to her in him.
Morragainsman: his that that is a mine.

LUCIO:
I waid the underer you: if way I home:
I am condemman! Come, it you 'tis bedrop:
There not says the diep in his remarden,
Marest for it a gentremain in ear.
Hath take me least a dexires, our head!
Our Restay one trucke, mind hath dows leards
His knoctorder to by, of tears.

HASTINGS:
O marcherer: maday all; stay parted and good,
When, the naid sound hather than beens
And and so tensixate, saying this made
That
With pulonsing pust, that me body more.

MENENIUS:
I am God good to at I tricking,
I guarlard ten to Romeo's, time say's that Bucking
good mortal into first. Sould serve in.